In [ ]:
import pyspark.sql.functions as F

In [ ]:
from spotify.utils import to_datestamp, to_date

In [ ]:
catalog = dbutils.widgets.get('catalog')

In [ ]:
checkpoint_path = f"abfss://{catalog}@dablake.dfs.core.windows.net/silver/spotify_events/_checkpoint/"

In [ ]:
df = (spark.readStream.table(f"{catalog}.bronze.spotify_events")
      .transform(lambda df: to_datestamp(df, "event_time", "event_timestamp"))
      .transform(lambda df: to_date(df, "event_time", "event_date"))
      .withColumn("played_in_mins", (F.col("ms_played") / 60000).cast("decimal(10,2)"))
      .withWatermark("event_timestamp", "30 minutes")
      .dropDuplicates(["event_id"])
      .writeStream
      .format("delta")
      .option("checkpointLocation", checkpoint_path)
      .trigger(availableNow=True)
      .toTable(f"{catalog}.silver.spotify_events")
)
df.awaitTermination()

In [ ]:
users = (spark.table(f"{catalog}.bronze.spotify_users")
          .withColumn('member_time', F.date_diff(F.current_date(), F.to_date('signup_date', 'yyyy-MM-dd')))
          
                      
          )

users.writeTo(f"{catalog}.silver.spotify_users").createOrReplace()

In [ ]:
tracks = (spark.table(f"{catalog}.bronze.spotify_tracks")
          .withColumn('track_age', F.date_diff(F.current_date(), F.to_date('release_date', 'yyyy-MM-dd')))
          .withColumn("duration_in_mins", (F.col("duration_ms") / 60000).cast("decimal(10,2)"))
).writeTo(f"{catalog}.silver.spotify_tracks").createOrReplace()